# Phase 1: 키워드 기반 필터링 (Keyword Filtering)

이 노트북은 LLM(Claude)을 통해 발견 및 확장된 키워드 사전을 사용하여 전체 데이터에서 UMC 관련 게시글만 필터링하는 과정을 수행합니다.

- **입력 데이터**: `data/processed/01_filtered_merged.csv` (원문에서 DELETED/BLOCKED 상태인 글이 제거된 초기 병합 데이터)
- **사용 키워드**: `config/keywords.yaml` (Phase 0에서 생성된 최종 UMC 키워드 사전)
- **출력 데이터**: `data/processed/01_keyword_filtered.csv` (키워드가 1개 이상 매칭된 게시글 모음)

이 노트북을 실행하기 전에 `config/keywords.yaml`에 확장된 키워드가 잘 저장되어 있는지 확인하세요.

In [ ]:
import os
import sys
from pathlib import Path

# 현재 작업 디렉토리를 프로젝트 루트로 설정
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(f"현재 작업 디렉토리: {Path.cwd()}")

sys.path.append(str(Path.cwd()))

## 1. 키워드 사전 확인

사용할 키워드가 제대로 로드되는지 확인합니다. 

In [ ]:
from src.filter_by_keyword import load_keywords

keywords = load_keywords("config/keywords.yaml")
print(f"총 {len(keywords)}개 차원이 로드되었습니다.\n")

for dim, kws in keywords.items():
    print(f"[{dim}]: {len(kws)}개 키워드")
    # 예시로 5개만 출력해 봅니다
    print(f"  예시: {', '.join(kws[:5])}{'...' if len(kws)>5 else ''}\n")

## 2. 키워드 필터링 실행

`src.filter_by_keyword`의 `run()` 함수를 호출하여 필터링을 수행합니다.
주의: 전체 데이터셋 크기에 따라 시간이 다소 소요될 수 있습니다. 진행 프로그레스바(tqdm)가 표시됩니다.

In [ ]:
from src.filter_by_keyword import run as run_keyword_filter

INPUT_FILE = "data/processed/01_cleaned_merged.csv"
OUTPUT_FILE = "data/processed/02_keyword_filtered.csv"

filtered_df = run_keyword_filter(
    input_path=INPUT_FILE,
    output_path=OUTPUT_FILE,
    keywords_path="config/keywords.yaml"
)

print(f"\n최종 저장된 데이터(키워드 매칭됨): {len(filtered_df):,}건")

## 3. 필터링 결과 확인

필터링된 데이터의 차원별 분포를 확인해 봅니다.

In [ ]:
# 각 게시글이 어떤 차원(복수 가능)에 매칭되었는지 확인
dim_counts = filtered_df['matched_dimensions'].value_counts()
print("상위 매칭 차원 조합 Top 10:\n")
print(dim_counts.head(10))